# Sesión 2: BigGAN | El zoológico de clases

**Arqueología de la I.A. generativa · Taller MUTAR**

**Año de referencia:** 2018 | **Modelo:** BigGAN (DeepMind)

---

### Antes de empezar

1. **Haz tu propia copia:** `Archivo → Guardar una copia en Drive`.
2. **Activa la GPU:** `Entorno de ejecución → Cambiar tipo de entorno de ejecución → T4 GPU → Guardar`.
3. **Ejecuta las celdas en orden**, de arriba hacia abajo (botón ▶ o `Ctrl + Enter`).
4. Las celdas con **formulario** se modifican sin tocar el código: cambia el valor a la derecha y vuelve a ejecutar solo esa celda.

**No necesitas saber programar.**

### Qué haremos hoy

1. Generar imágenes de una categoría que elijas, entre las 1000 de ImageNet.
2. Ajustar la **truncación** para elegir entre imágenes más típicas o más extrañas.
3. Crear **híbridos** entre dos categorías: una imagen que es parte iglesia y parte plátano.
4. Hacer un **video** que viaja de una categoría a otra, o por muchas categorías seguidas.

**Te llevas:** imágenes de tu clase favorita y un video de interpolación entre clases. Los vamos a usar en la Sesión 4 (FILM).

## ¿Qué es BigGAN?

En la Sesión 1 usamos una red que **reconoce** imágenes y la obligamos a "soñar" sobre una foto. Hoy usamos una red que **genera** imágenes desde cero.

**GAN** significa *Generative Adversarial Network*, red generativa adversaria (Goodfellow y equipo, 2014). Son dos redes que se entrenan juntas, como un juego:

* El **generador** recibe números al azar y tiene que convertirlos en una imagen.
* El **discriminador** recibe imágenes, algunas reales y otras del generador, y tiene que adivinar cuáles son falsas.

Cada una mejora intentando ganarle a la otra. Al final del entrenamiento nos quedamos solo con el generador.

**BigGAN** (Andrew Brock, Jeff Donahue y Karen Simonyan, DeepMind, 2018) fue un salto de escala: redes y lotes de entrenamiento mucho más grandes que los de entonces, entrenados con **ImageNet**, el mismo conjunto de 1000 categorías con el que se entrenó la red de la Sesión 1. Por eso, otra vez, aparecen tantos perros.

Es una GAN **condicional por clase**: además de los números al azar, le decimos qué categoría queremos.

> Nota técnica: usamos **BigGAN-deep**, una variante más profunda presentada en el mismo artículo. Los pesos entrenados los publicó DeepMind y los convirtió a PyTorch el equipo de Hugging Face. Genera imágenes cuadradas de 128, 256 o 512 píxeles.

### Las tres perillas

* **Clase:** qué categoría generar. Es un vector de 1000 posiciones con una sola encendida.
* **Ruido (semilla):** 128 números al azar que deciden *cómo* es esa imagen: pose, encuadre, colores. La **semilla** fija esos números: misma semilla, misma imagen.
* **Truncación:** los 128 números se sacan de una campana de probabilidades. La truncación recorta los valores más extremos. Con **truncación baja** quedan cerca del centro y salen imágenes típicas, nítidas y parecidas entre sí. Con **truncación alta** (hasta 1) hay más variedad, más rarezas y más alucinación.

In [ ]:
#@title 0. Preparar el entorno (ejecutar primero)
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pytorch-pretrained-biggan", "imageio-ffmpeg"], check=True)

import os, io, time, glob, shutil, json
import numpy as np
import torch
import requests
from PIL import Image, ImageDraw, ImageFont
from IPython.display import display, Video
from tqdm.auto import tqdm
from google.colab import files, drive
from pytorch_pretrained_biggan import BigGAN, truncated_noise_sample

dispositivo = "cuda" if torch.cuda.is_available() else "cpu"
if dispositivo == "cuda":
    print("GPU activada:", torch.cuda.get_device_name(0))
else:
    print("ATENCIÓN: no hay GPU activa. Funciona, pero mucho más lento.")
    print("Ve a Entorno de ejecución → Cambiar tipo de entorno de ejecución → T4 GPU → Guardar,")
    print("y vuelve a ejecutar esta celda.")

CARPETA = "/content/resultados/sesion2"
os.makedirs(CARPETA, exist_ok=True)

### 0.1 Guardar en tu Google Drive (recomendado)

Así todo lo que generes queda guardado en tu Drive, ordenado por sesión, aunque se cierre Colab:

```
Mi unidad/
└── Taller_MUTAR/
    ├── sesion1/   ← DeepDream
    ├── sesion2/   ← esta sesión
    └── ...
```

Es la misma carpeta `Taller_MUTAR` que usamos en la Sesión 1. Si prefieres no conectar Drive, desmarca la casilla: los resultados quedan en Colab y los descargas al final (sección 7).

In [ ]:
#@title 0.1 Elegir dónde guardar
GUARDAR_EN_DRIVE = True #@param {type:"boolean"}
CARPETA_TALLER = "/content/drive/MyDrive/Taller_MUTAR" #@param {type:"string"}

if GUARDAR_EN_DRIVE:
    try:
        drive.mount("/content/drive")
        CARPETA = os.path.join(CARPETA_TALLER, "sesion2")
        os.makedirs(CARPETA, exist_ok=True)
    except Exception as e:
        print("No se pudo conectar Drive (permite las ventanas emergentes y usa la misma cuenta de Google que en Colab).")
        print("Los resultados se guardarán solo en Colab:", e)
print("Los resultados se guardarán en:", CARPETA)

## 1. Cargar BigGAN

Elige la **resolución**. 256 es el equilibrio recomendado. 512 da imágenes cuatro veces más grandes (en píxeles), pero tarda unas cuatro veces más y la primera descarga pesa más.

Si cambias la resolución, vuelve a ejecutar esta celda.

In [ ]:
#@title 1. Cargar el modelo
RESOLUCION = 256 #@param [128, 256, 512] {type:"raw"}

print(f"Descargando BigGAN-deep {RESOLUCION}... (la primera vez puede tardar un par de minutos)")
modelo = BigGAN.from_pretrained(f"biggan-deep-{RESOLUCION}").to(dispositivo).eval()
LOTE = {128: 16, 256: 8, 512: 4}[RESOLUCION]   # cuántas imágenes se generan a la vez
print(f"Modelo cargado. Genera imágenes de {RESOLUCION} x {RESOLUCION} px.")

## 2. El catálogo de 1000 clases

BigGAN conoce las mismas 1000 categorías de ImageNet. Los nombres están **en inglés**.

En todos los formularios puedes escribir la clase **por su nombre** (`church`, `monarch butterfly`) **o por su número** (`497`). Si escribes una parte del nombre (`butterfly`), el notebook busca coincidencias.

Para explorar, usa el **buscador** de la celda 2.1: puedes buscar por palabra, por número, por rango de números (`300-340`) o ver las 1000 de una vez. Las clases vecinas suelen ser parecidas (del 0 al 397 son animales; del 398 en adelante, objetos, comidas, plantas y paisajes). Ojo: unas 120 clases son razas de perro y aparecen por el nombre de la raza (`Labrador Retriever`, `Pug`, `Dalmatian`), así que buscar "dog" no las encuentra a todas.

In [ ]:
#@title 2. Cargar el catálogo de clases
URL_CLASES = "https://raw.githubusercontent.com/anishathalye/imagenet-simple-labels/master/imagenet-simple-labels.json"
nombres_clases = requests.get(URL_CLASES, timeout=30).json()
nombre_a_indice = {n.lower(): i for i, n in enumerate(nombres_clases)}

def resolver_clase(texto):
    # Acepta un número (0-999), un nombre exacto o una parte del nombre.
    t = str(texto).strip().lower()
    if t.isdigit():
        n = int(t)
        if 0 <= n < 1000:
            return n
        raise SystemExit(f"El número {n} no es válido: debe estar entre 0 y 999.")
    if t in nombre_a_indice:
        return nombre_a_indice[t]
    parecidas = [i for i, n in enumerate(nombres_clases) if t in n.lower()]
    if len(parecidas) == 1:
        return parecidas[0]
    if parecidas:
        lista = ", ".join(f"{nombres_clases[i]} ({i})" for i in parecidas[:15])
        raise SystemExit(f"'{texto}' coincide con varias clases: {lista}. Escribe una exacta.")
    raise SystemExit(f"No encontré '{texto}'. Usa el buscador (celda 2.1) o escribe el nombre en inglés.")

def nombre(i):
    return nombres_clases[i]

print(f"Clases disponibles: {len(nombres_clases)}")
print("Algunas para empezar:")
for i in [497, 323, 954, 2, 417, 980, 107, 84, 483, 437, 973, 327]:
    print(f"  {i:>3}  {nombre(i)}")

In [ ]:
#@title 2.1 Buscador de clases
#@markdown Escribe una **palabra** (`dog`, `bird`), un **número** (`497`), un **rango** (`300-340`) o **todas** para ver el catálogo completo.
BUSCAR = "300-340" #@param {type:"string"}

t = BUSCAR.strip().lower()
if t in ("todas", "todo", "all", ""):
    encontradas = list(enumerate(nombres_clases))
    print("Catálogo completo (1000 clases):")
elif "-" in t and t.replace("-", "").replace(" ", "").isdigit():
    a, b = sorted(int(x) for x in t.split("-")[:2])
    a, b = max(0, a), min(999, b)
    encontradas = [(i, nombres_clases[i]) for i in range(a, b + 1)]
    print(f"Clases del {a} al {b}:")
elif t.isdigit():
    n = int(t)
    if not 0 <= n < 1000:
        raise SystemExit("El número tiene que estar entre 0 y 999.")
    encontradas = [(i, nombres_clases[i]) for i in range(max(0, n - 5), min(999, n + 5) + 1)]
    print(f"La clase {n} es «{nombres_clases[n]}». Sus vecinas:")
else:
    encontradas = [(i, n) for i, n in enumerate(nombres_clases) if t in n.lower()]
    print(f"{len(encontradas)} clases contienen «{BUSCAR}»:")

# Se imprime en columnas para que entre más en pantalla
filas = [f"{i:>3}  {n[:28]:<28}" for i, n in encontradas]
columnas = 3 if len(filas) > 30 else 1
alto = (len(filas) + columnas - 1) // columnas
for k in range(alto):
    print("   ".join(filas[k + c * alto] for c in range(columnas) if k + c * alto < len(filas)))

## 3. El motor de generación

Esta celda define las funciones que convierten números en imágenes. No hace falta modificarla: solo ejecútala.

* `ruido(semilla)`: los 128 números al azar de una semilla.
* `vector_clase(...)`: la clase, o una mezcla de clases.
* `generar(...)`: le pasa todo a BigGAN y devuelve imágenes.

In [ ]:
#@title 3. Motor de generación (solo ejecutar)
def ruido(semilla, truncacion):
    # 128 números al azar, sacados de una campana recortada (truncada).
    z = truncated_noise_sample(truncation=truncacion, batch_size=1, seed=int(semilla))
    return torch.from_numpy(z)[0]

def vector_clase(mezcla):
    # mezcla: {indice_de_clase: peso}. Ej: {497: 0.7, 954: 0.3} = 70% iglesia, 30% plátano.
    y = torch.zeros(1000)
    for i, p in mezcla.items():
        y[i] += p
    return y

def a_imagen(t):
    # Tensor de BigGAN (-1 a 1) → imagen visible.
    a = ((t.clamp(-1, 1) + 1) * 127.5).round().byte().permute(1, 2, 0).cpu().numpy()
    return Image.fromarray(a)

@torch.no_grad()
def generar(zs, ys, truncacion):
    # zs: lista de vectores de ruido · ys: lista de vectores de clase · devuelve lista de imágenes
    if not (0 < truncacion <= 1):
        raise SystemExit("La truncación tiene que ser mayor que 0 y como máximo 1.")
    salida = []
    for k in range(0, len(zs), LOTE):
        z = torch.stack(zs[k:k + LOTE]).to(dispositivo)
        y = torch.stack(ys[k:k + LOTE]).to(dispositivo)
        salida += [a_imagen(t) for t in modelo(z, y, truncacion)]
    return salida

def slerp(a, b, t):
    # Interpolación "esférica" entre dos ruidos: más suave que una línea recta.
    an, bn = a / a.norm(), b / b.norm()
    omega = torch.acos((an * bn).sum().clamp(-1, 1))
    if omega.abs() < 1e-4:
        return (1 - t) * a + t * b
    return (torch.sin((1 - t) * omega) * a + torch.sin(t * omega) * b) / torch.sin(omega)

def suavizar(t):
    # Arranca y frena despacio (en vez de velocidad constante).
    return t * t * (3 - 2 * t)

def grilla(imagenes, etiquetas=None, columnas=4, lado=256):
    # Junta varias imágenes en una sola, con etiquetas debajo.
    filas = (len(imagenes) + columnas - 1) // columnas
    alto_txt = 28 if etiquetas else 0
    hoja = Image.new("RGB", (columnas * lado, filas * (lado + alto_txt)), "white")
    d = ImageDraw.Draw(hoja)
    try:
        from matplotlib import font_manager
        fuente = ImageFont.truetype(font_manager.findfont("DejaVu Sans"), 15)
    except Exception:
        fuente = ImageFont.load_default()
    for k, im in enumerate(imagenes):
        x, y = (k % columnas) * lado, (k // columnas) * (lado + alto_txt)
        hoja.paste(im.resize((lado, lado), Image.LANCZOS), (x, y))
        if etiquetas:
            d.text((x + 6, y + lado + 5), etiquetas[k], fill=(0, 0, 0), font=fuente)
    return hoja

def guardar(img, nombre_archivo):
    ruta = os.path.join(CARPETA, nombre_archivo)
    img.save(ruta)
    print("Guardado:", ruta)
    return ruta

def limpio(texto):
    return "".join(c if c.isalnum() else "_" for c in texto)

_modelos = {}
def modelo_de(res):
    # Devuelve el modelo de esa resolución; si no es el que está cargado, lo carga una sola vez.
    if res == RESOLUCION:
        return modelo
    if res not in _modelos:
        print(f"Cargando BigGAN-deep {res} (solo la primera vez, puede tardar un poco)...")
        _modelos[res] = BigGAN.from_pretrained(f"biggan-deep-{res}").to(dispositivo).eval()
    return _modelos[res]

def etiqueta_modo(modo):
    return "512" if modo.startswith("Regenerar") else ("x4" if "×4" in modo else "x2")

@torch.no_grad()
def en_grande(z, y, truncacion, modo, nombre_archivo):
    # Genera una sola imagen en grande: ampliando la misma imagen o regenerándola a 512 px.
    if modo.startswith("Regenerar"):
        m = modelo_de(512)
        img = a_imagen(m(z[None].to(dispositivo), y[None].to(dispositivo), truncacion)[0])
        if RESOLUCION != 512:
            print("Ojo: el modelo de 512 es otra red. La imagen será parecida, pero no idéntica.")
    else:
        img = generar([z], [y], truncacion)[0]
        f = 4 if "×4" in modo else 2
        img = img.resize((img.width * f, img.height * f), Image.LANCZOS)
    print(f"Tamaño: {img.width} x {img.height} px")
    display(img)
    return guardar(img, nombre_archivo)

print("Motor listo.")

## 4. Tu primera imagen

Elige una clase, una truncación y una semilla. Con **CANTIDAD** generas varias imágenes de la misma clase de una vez, cada una con la semilla siguiente (42, 43, 44…).

**Ideas:** church, monarch butterfly, banana, volcano, jellyfish, peacock, castle, lighthouse, coral reef, starfish.

In [ ]:
#@title 4. Generar imágenes de una clase
CLASE = "church" #@param {type:"string"}
TRUNCACION = 0.4 #@param {type:"slider", min:0.05, max:1.0, step:0.05}
SEMILLA = 42 #@param {type:"integer"}
CANTIDAD = 4 #@param {type:"slider", min:1, max:8, step:1}

c = resolver_clase(CLASE)
semillas = list(range(SEMILLA, SEMILLA + CANTIDAD))
imgs = generar([ruido(s, TRUNCACION) for s in semillas], [vector_clase({c: 1})] * CANTIDAD, TRUNCACION)

print(f"Clase {c}: {nombre(c)} · truncación {TRUNCACION}")
display(grilla(imgs, [f"semilla {s}" for s in semillas], columnas=min(CANTIDAD, 4)))
for s, im in zip(semillas, imgs):
    guardar(im, f"biggan_{limpio(nombre(c))}_t{TRUNCACION}_s{s}.png")

### 4.1 Comparar truncaciones

La misma clase y la misma semilla con seis truncaciones distintas, de izquierda a derecha: de lo más típico a lo más extraño.

In [ ]:
#@title 4.1 Comparar truncaciones
CLASE_T = "church" #@param {type:"string"}
SEMILLA_T = 42 #@param {type:"integer"}

c = resolver_clase(CLASE_T)
valores = [0.1, 0.25, 0.4, 0.6, 0.8, 1.0]
imgs = [generar([ruido(SEMILLA_T, t)], [vector_clase({c: 1})], t)[0] for t in valores]
hoja = grilla(imgs, [f"truncación {t}" for t in valores], columnas=6)
display(hoja)
guardar(hoja, f"truncaciones_{limpio(nombre(c))}_s{SEMILLA_T}.png")

### 4.2 Sacar una truncación en grande

¿Te gustó una de las seis? Elige cuál y cómo agrandarla. Usa la misma clase y semilla de la celda 4.1.

* **Ampliar ×2 o ×4:** la misma imagen exacta, más grande. No agrega detalle.
* **Regenerar a 512 px:** más detalle real, pero el modelo de 512 es otra red: la imagen será parecida, no idéntica. La primera vez tarda un poco más porque descarga ese modelo.

In [ ]:
#@title 4.2 Una truncación en grande
TRUNCACION_ELEGIDA = 1.0 #@param [0.1, 0.25, 0.4, 0.6, 0.8, 1.0] {type:"raw"}
MODO = "Ampliar ×2" #@param ["Ampliar ×2", "Ampliar ×4", "Regenerar a 512 px (más detalle)"]

c = resolver_clase(CLASE_T)
en_grande(ruido(SEMILLA_T, TRUNCACION_ELEGIDA), vector_clase({c: 1}), TRUNCACION_ELEGIDA, MODO,
          f"grande_{limpio(nombre(c))}_t{TRUNCACION_ELEGIDA}_s{SEMILLA_T}_{etiqueta_modo(MODO)}.png")

## 5. Híbridos: mezclar dos clases

El vector de clase no tiene por qué ser una sola categoría: podemos pedirle a BigGAN, por ejemplo, **70% iglesia y 30% plátano**. El resultado no es un collage de dos fotos, sino una criatura que no existe en ningún dataset.

La celda muestra cinco pasos de la mezcla, de 100% A a 100% B.

In [ ]:
#@title 5. Híbrido entre dos clases
CLASE_A = "church" #@param {type:"string"}
CLASE_B = "banana" #@param {type:"string"}
TRUNCACION_H = 0.4 #@param {type:"slider", min:0.05, max:1.0, step:0.05}
SEMILLA_H = 123 #@param {type:"integer"}

a, b = resolver_clase(CLASE_A), resolver_clase(CLASE_B)
pesos = [0, 0.25, 0.5, 0.75, 1]
z = ruido(SEMILLA_H, TRUNCACION_H)
imgs = generar([z] * 5, [vector_clase({a: 1 - p, b: p}) for p in pesos], TRUNCACION_H)
hoja = grilla(imgs, [f"{int((1 - p) * 100)}% A · {int(p * 100)}% B" for p in pesos], columnas=5)
print(f"A = {nombre(a)} · B = {nombre(b)}")
display(hoja)
guardar(hoja, f"hibrido_{limpio(nombre(a))}_{limpio(nombre(b))}_s{SEMILLA_H}.png")
guardar(imgs[2], f"hibrido_50_{limpio(nombre(a))}_{limpio(nombre(b))}_s{SEMILLA_H}.png")

### 5.1 Sacar un híbrido en grande

¿Te gustó una de las cinco mezclas? Elige cuál (el porcentaje de B) y cómo agrandarla. Usa las mismas clases, semilla y truncación de la celda 5. También puedes elegir una proporción que no estaba en la tira, como 60%.

In [ ]:
#@title 5.1 Un híbrido en grande
PORCENTAJE_B = 50 #@param {type:"slider", min:0, max:100, step:5}
MODO_H = "Ampliar ×2" #@param ["Ampliar ×2", "Ampliar ×4", "Regenerar a 512 px (más detalle)"]

a, b = resolver_clase(CLASE_A), resolver_clase(CLASE_B)
p = PORCENTAJE_B / 100
print(f"{100 - PORCENTAJE_B}% {nombre(a)} · {PORCENTAJE_B}% {nombre(b)}")
en_grande(ruido(SEMILLA_H, TRUNCACION_H), vector_clase({a: 1 - p, b: p}), TRUNCACION_H, MODO_H,
          f"grande_hibrido_{limpio(nombre(a))}_{limpio(nombre(b))}_{PORCENTAJE_B}_s{SEMILLA_H}_{etiqueta_modo(MODO_H)}.png")

## 6. Video: viajar entre clases

Un video es una secuencia de imágenes (*frames*). **Duración × FPS = cantidad de frames.** 10 segundos a 24 FPS son 240 imágenes.

BigGAN es rápida: con GPU genera decenas de imágenes por segundo a 256 px. Por eso se pueden hacer **videos largos** sin esperar demasiado.

| Resolución | 10 s a 24 FPS (240 frames, GPU T4, aprox.) |
|---|---|
| 128 | menos de 30 s |
| 256 | 30 s a 1 min |
| 512 | 2 a 4 min |

La celda imprime una estimación real apenas empieza.

**Para que el video sea más grande** hay dos caminos:

* **Resolución 512** en la celda 1: es la máxima que genera BigGAN, con detalle real.
* **AMPLIAR** (celda 6.0): agranda cada frame al doble o al cuádruple. Sirve para proyectar en pantallas grandes, pero **no agrega detalle**: solo estira la imagen con suavidad.

Los frames se guardan como imágenes en una carpeta. Sirven para la **Sesión 4 (FILM)**, que puede inventar frames intermedios y hacer el video aún más fluido.

In [ ]:
#@title 6.0 Formato del video (ejecutar antes de 6.1 y 6.2)
FPS = 24 #@param [12, 24, 25, 30] {type:"raw"}
AMPLIAR = 1 #@param [1, 2, 4] {type:"raw"}
#@markdown CALIDAD: 18 = muy alta (archivo grande), 23 = buena, 28 = liviana (para WhatsApp)
CALIDAD = 18 #@param {type:"slider", min:14, max:30, step:1}

def ffmpeg():
    return shutil.which("ffmpeg") or __import__("imageio_ffmpeg").get_ffmpeg_exe()

def renderizar(nombre_video, claves, segundos_por_tramo, truncacion, suave=True, ida_y_vuelta=False, gif=False):
    # claves: lista de (vector_de_ruido, vector_de_clase). El video viaja de una a la siguiente.
    carpeta = os.path.join(CARPETA, f"frames_{nombre_video}")
    shutil.rmtree(carpeta, ignore_errors=True)
    os.makedirs(carpeta)
    por_tramo = max(2, int(round(segundos_por_tramo * FPS)))
    zs, ys = [], []
    for (z0, y0), (z1, y1) in zip(claves[:-1], claves[1:]):
        for k in range(por_tramo):
            t = k / por_tramo
            t = suavizar(t) if suave else t
            zs.append(slerp(z0, z1, t))
            ys.append((1 - t) * y0 + t * y1)
    zs.append(claves[-1][0]); ys.append(claves[-1][1])
    total = len(zs)
    lado = RESOLUCION * AMPLIAR
    print(f"{total} frames · {total / FPS:.1f} s · {lado} x {lado} px")
    inicio, n = time.time(), 0
    for k in tqdm(range(0, total, LOTE), desc="Generando"):
        for im in generar(zs[k:k + LOTE], ys[k:k + LOTE], truncacion):
            if AMPLIAR > 1:
                im = im.resize((lado, lado), Image.LANCZOS)
            im.save(os.path.join(carpeta, f"{n:06d}.jpg"), quality=95)
            n += 1
        if k == 0 and total > LOTE:
            seg = (time.time() - inicio) / LOTE * (total - LOTE)
            print(f"Tiempo estimado restante: {seg / 60:.1f} min")
    return montar_video(nombre_video, ida_y_vuelta=ida_y_vuelta, gif=gif)

def montar_video(nombre_video, fps=None, ida_y_vuelta=False, calidad=None, gif=False, mostrar=True):
    # Une los frames de la carpeta en un .mp4 (y opcionalmente un .gif liviano).
    fps, calidad = fps or FPS, calidad or CALIDAD
    frames = sorted(glob.glob(os.path.join(CARPETA, f"frames_{nombre_video}", "*.jpg")))
    if not frames:
        raise SystemExit(f"No hay frames para '{nombre_video}'.")
    if ida_y_vuelta:
        frames = frames + frames[-2:0:-1]
    tmp = "/content/_montaje"
    shutil.rmtree(tmp, ignore_errors=True); os.makedirs(tmp)
    for i, f in enumerate(frames):
        os.symlink(f, os.path.join(tmp, f"{i:06d}.jpg"))
    salida = os.path.join(CARPETA, f"{nombre_video}.mp4")
    subprocess.run([ffmpeg(), "-y", "-loglevel", "error", "-framerate", str(fps), "-i", f"{tmp}/%06d.jpg",
                    "-c:v", "libx264", "-preset", "medium", "-crf", str(calidad),
                    "-pix_fmt", "yuv420p", "-movflags", "+faststart", salida], check=True)
    mb = os.path.getsize(salida) / 1e6
    print(f"Video: {salida} · {len(frames)} frames · {len(frames) / fps:.1f} s · {mb:.1f} MB")
    if gif:
        salida_gif = os.path.join(CARPETA, f"{nombre_video}.gif")
        subprocess.run([ffmpeg(), "-y", "-loglevel", "error", "-i", salida, "-vf",
                        "fps=12,scale=384:-2:flags=lanczos,split[a][b];[a]palettegen[p];[b][p]paletteuse",
                        salida_gif], check=True)
        print(f"GIF: {salida_gif} · {os.path.getsize(salida_gif) / 1e6:.1f} MB")
    if mostrar and mb < 40:
        display(Video(salida, embed=True, width=512))
    elif mostrar:
        print("(El video es muy pesado para mostrarlo aquí: descárgalo en la sección 7 o búscalo en tu Drive.)")
    return salida

print(f"Formato listo: {FPS} FPS · frames de {RESOLUCION * AMPLIAR} x {RESOLUCION * AMPLIAR} px")

### 6.1 De una clase a otra

El video empieza en la clase A y termina en la clase B. En el camino aparecen los híbridos.

**Las dos semillas.** El video tiene dos extremos y cada uno tiene su propio ruido:

* **SEMILLA_A:** el ruido del **primer frame** (clase A). Decide la pose, el encuadre y los colores con los que empieza.
* **SEMILLA_B:** el ruido del **último frame** (clase B). Decide cómo termina.

| SEMILLA_B | qué cambia en el video |
|---|---|
| vacía (por defecto) | solo **qué es**: el ruido queda fijo, la pose se mantiene y la medusa se vuelve estrella "en el mismo lugar" |
| otro número | **qué es** y **cómo es**: cambian la clase y también la pose, el encuadre y el fondo |

Consejo: prueba primero con SEMILLA_B vacía y después con un número, para ver la diferencia.

* **SUAVIZADO:** el viaje arranca y frena despacio, en vez de ir a velocidad constante.
* **IDA_Y_VUELTA:** vuelve de B a A. El video queda del doble de largo y se puede repetir en bucle.

**Combinaciones para probar:** church → banana · monarch butterfly → great white shark · volcano → jellyfish · peacock → castle · lighthouse → balloon.

In [ ]:
#@title 6.1 Video entre dos clases
NOMBRE_VIDEO = "iglesia_platano" #@param {type:"string"}
CLASE_A_V = "church" #@param {type:"string"}
CLASE_B_V = "banana" #@param {type:"string"}
DURACION_SEG = 8 #@param {type:"slider", min:2, max:60, step:1}
TRUNCACION_V = 0.4 #@param {type:"slider", min:0.05, max:1.0, step:0.05}
#@markdown SEMILLA_A = ruido del primer frame · SEMILLA_B = ruido del último (vacía = el mismo que A: solo cambia la clase)
SEMILLA_A = 123 #@param {type:"integer"}
SEMILLA_B = "" #@param {type:"string"}
SUAVIZADO = True #@param {type:"boolean"}
IDA_Y_VUELTA = True #@param {type:"boolean"}
HACER_GIF = False #@param {type:"boolean"}

a, b = resolver_clase(CLASE_A_V), resolver_clase(CLASE_B_V)
za = ruido(SEMILLA_A, TRUNCACION_V)
zb = ruido(int(SEMILLA_B), TRUNCACION_V) if SEMILLA_B.strip() else za
print(f"{nombre(a)} → {nombre(b)}")
renderizar(limpio(NOMBRE_VIDEO), [(za, vector_clase({a: 1})), (zb, vector_clase({b: 1}))],
           DURACION_SEG, TRUNCACION_V, suave=SUAVIZADO, ida_y_vuelta=IDA_Y_VUELTA, gif=HACER_GIF)

### 6.2 Viaje largo por varias clases

Para videos largos: escribe un **recorrido** de clases separadas por comas y el video pasa por todas, una tras otra. Con **VOLVER_AL_INICIO** la última clase regresa a la primera y el video queda en bucle perfecto.

* **SEGUNDOS_POR_TRAMO:** cuánto dura cada viaje entre dos clases. 6 clases × 5 segundos = 30 segundos.
* **SEMILLA_VJ:** una sola semilla (un solo ruido) para todo el recorrido. La pose se mantiene y solo cambian las clases.
* **SEMILLAS:** opcional. Una semilla por clase, separadas por comas (`7, 42, 99…`, tantas como clases). Así cada tramo cambia también de forma. Si lo dejas vacío, se usa SEMILLA_VJ para todas.

Ejemplo de recorrido: `church, banana, jellyfish, volcano, peacock, castle`

In [ ]:
#@title 6.2 Viaje largo por varias clases
NOMBRE_VIAJE = "viaje" #@param {type:"string"}
RECORRIDO = "church, banana, jellyfish, volcano, peacock, castle" #@param {type:"string"}
SEGUNDOS_POR_TRAMO = 5 #@param {type:"slider", min:1, max:30, step:1}
TRUNCACION_VJ = 0.4 #@param {type:"slider", min:0.05, max:1.0, step:0.05}
#@markdown SEMILLA_VJ = un solo ruido para todo el viaje · SEMILLAS = una por clase, separadas por comas (opcional)
SEMILLA_VJ = 7 #@param {type:"integer"}
SEMILLAS = "" #@param {type:"string"}
VOLVER_AL_INICIO = True #@param {type:"boolean"}
HACER_GIF_VJ = False #@param {type:"boolean"}

clases = [resolver_clase(x) for x in RECORRIDO.split(",") if x.strip()]
if len(clases) < 2:
    raise SystemExit("Escribe al menos dos clases separadas por comas.")
lista_semillas = [int(s) for s in SEMILLAS.split(",") if s.strip()] or [SEMILLA_VJ] * len(clases)
if len(lista_semillas) != len(clases):
    raise SystemExit(f"Escribiste {len(lista_semillas)} semillas para {len(clases)} clases: tienen que ser la misma cantidad.")
claves = [(ruido(s, TRUNCACION_VJ), vector_clase({c: 1})) for c, s in zip(clases, lista_semillas)]
if VOLVER_AL_INICIO:
    claves.append(claves[0])
print(" → ".join(nombre(c) for c in clases) + (" → " + nombre(clases[0]) if VOLVER_AL_INICIO else ""))
renderizar(limpio(NOMBRE_VIAJE), claves, SEGUNDOS_POR_TRAMO, TRUNCACION_VJ, suave=True, gif=HACER_GIF_VJ)

In [ ]:
#@title 6.3 (Opcional) Volver a montar un video con otros ajustes
#@markdown Cambia FPS, calidad, ida y vuelta o GIF sin volver a generar los frames.
#@markdown Ejemplo: los mismos frames a 12 FPS dan un video del doble de largo y más lento.
NOMBRE = "iglesia_platano" #@param {type:"string"}
FPS_MONTAJE = 24 #@param [8, 12, 15, 24, 25, 30] {type:"raw"}
IDA_Y_VUELTA_M = False #@param {type:"boolean"}
CALIDAD_M = 18 #@param {type:"slider", min:14, max:30, step:1}
HACER_GIF_M = True #@param {type:"boolean"}

montar_video(limpio(NOMBRE), fps=FPS_MONTAJE, ida_y_vuelta=IDA_Y_VUELTA_M, calidad=CALIDAD_M, gif=HACER_GIF_M)

## 7. Descargar tus resultados

Descarga un `.zip` con imágenes, videos y GIFs. Los frames sueltos no se incluyen por defecto porque pesan mucho; si los quieres para la Sesión 4, marca INCLUIR_FRAMES. Si conectaste Drive (celda 0.1), ya están todos guardados en `Taller_MUTAR/sesion2`.

In [ ]:
#@title 7. Descargar todo en un .zip
INCLUIR_FRAMES = False #@param {type:"boolean"}

tmp = "/content/_para_zip"  # copia temporal para armar el zip
shutil.rmtree(tmp, ignore_errors=True)
shutil.copytree(CARPETA, tmp, ignore=None if INCLUIR_FRAMES else shutil.ignore_patterns("frames_*"))
ruta_zip = shutil.make_archive("/content/sesion2_biggan", "zip", tmp)
print(f"{ruta_zip} ({os.path.getsize(ruta_zip) / 1e6:.1f} MB)")
if os.path.getsize(ruta_zip) > 500e6:
    print("Es muy pesado para descargar desde el navegador: conecta Drive en la celda 0.1, ahí ya queda todo guardado.")
files.download(ruta_zip)

---
## Glosario

* **GAN:** dos redes que se entrenan compitiendo. El generador crea imágenes y el discriminador intenta descubrir cuáles son falsas.
* **Clase:** la categoría que le pedimos a BigGAN (iglesia, plátano…). ImageNet tiene 1000.
* **Ruido:** 128 números al azar que deciden cómo es la imagen dentro de esa clase.
* **Semilla:** el número que fija ese ruido. Misma semilla + misma clase + misma truncación = misma imagen.
* **Truncación:** un valor mayor que 0 y hasta 1. Bajo: imágenes típicas y nítidas, poca variedad. Alto: más variedad, más rareza.
* **Espacio latente:** el espacio de todos los ruidos posibles. Cada punto es una imagen; moverse por él es transformar la imagen de a poco.
* **Interpolación:** viajar de un punto a otro en pasos pequeños. Aquí, entre clases, entre semillas o las dos cosas a la vez.

---
### Para la Sesión 4

Guarda al menos:

1. Una imagen de tu clase favorita (prueba distintas truncaciones y semillas).
2. Un video de interpolación entre dos clases que te interesen.

Los vamos a usar en la Sesión 4 (FILM) para crear transiciones entre las imágenes de todo el taller.

### Para seguir explorando

* Artículo original: Brock, Donahue y Simonyan, *Large Scale GAN Training for High Fidelity Natural Image Synthesis* (2018): https://arxiv.org/abs/1809.11096
* Implementación en PyTorch (Hugging Face): https://github.com/huggingface/pytorch-pretrained-biggan
* Implementación del autor: https://github.com/ajbrock/BigGAN-PyTorch

*Taller Arqueología de la I.A. generativa · Máquinas de Estados Infinitos*